In [ ]:
%load_ext autoreload
%autoreload 2

import scanpy as sc

adata = sc.read_h5ad("./data/B_cell/processed/bcell_velocity_processed_all_lanes.h5ad")
adata

In [ ]:
adata.obs["celltype_level2"]

In [ ]:
import flowmap
from flowmap import *

X = adata.X
V = adata.layers["velocity"]

emb = VectorFieldEmbedder(X, V, method="umap", dist_method="phase", alpha=0.5,
                          dof=30, knn_k=30,
                          embed_kwargs={"n_neighbors":30,
                                        "min_dist":0.5})
emb.fit_embedding(seed=42)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from flowmap.utils import compute_velocity_on_grid


# ------------------------------------------------------------
# Helper: remove grid seeds outside the manifold
# ------------------------------------------------------------
def points_inside_mask(X_emb, seeds, k=8, radius_scale=1.2):
    nn = NearestNeighbors(n_neighbors=k).fit(X_emb)
    r = np.median(nn.kneighbors(X_emb)[0][:, -1]) * radius_scale
    neigh_idx = nn.radius_neighbors(seeds, radius=r, return_distance=False)
    return np.array([len(ix) > 0 for ix in neigh_idx])


# ------------------------------------------------------------
# Data
# ------------------------------------------------------------
X_emb = emb.X_emb
celltypes = adata.obs["celltype_level1"].astype("category")
codes = celltypes.cat.codes
categories = celltypes.cat.categories
# velocity_pseudotime = adata.obs["velocity_pseudotime"]

# ------------------------------------------------------------
# Velocity grid (sparser)
# ------------------------------------------------------------
Xg, keep_mass, Vg = compute_velocity_on_grid(
    X_emb,
    spline_vf=emb.spline_vf,
    grid_size=20,          # ↓ sparser grid
    min_mass=0.01
)

keep_inside = points_inside_mask(X_emb, Xg)
Xg = Xg[keep_inside]
Vg = Vg[keep_inside]


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 8))
cmap = plt.get_cmap("tab20", len(categories))  # good for up to ~20 types
# scatter (larger + more transparent)
sc = ax.scatter(
    X_emb[:, 0],
    X_emb[:, 1],
    # c=velocity_pseudotime,
    # cmap="viridis",
    c=codes,
    cmap=cmap,
    s=100,
    alpha=0.1,
    linewidths=0,
)

# velocity arrows (thicker + bigger heads)
ax.quiver(
    Xg[:, 0], Xg[:, 1],
    Vg[:, 0], Vg[:, 1],
    angles="xy",
    scale_units="xy",
    scale=15,
    width=0.005,         # ↑ thicker
    headwidth=6.0,       # ↑ bigger head
    headlength=6.0,
    headaxislength=4.0,
    minlength=0.2,
    color="k",
    alpha=0.9,
)


# ------------------------------------------------------------
# Styling (no legend / no colorbar)
# ------------------------------------------------------------
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig(
    "./figures/bcell/bcell_velocity_stream.pdf",  # updated path
    dpi=400,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from flowmap.geometry.curvature import compute_flow_curvature

curv = compute_flow_curvature(emb)

In [ ]:
X = emb.X_emb

k_total  = curv["curvature"]["total"]
k_geod   = curv["curvature"]["geodesic"]
k_normal = curv["curvature"]["normal"]

def clip_quantile(arr, q_low=2, q_high=98):
    lo, hi = np.percentile(arr, [q_low, q_high])
    return np.clip(arr, lo, hi)

k_total_c  = clip_quantile(k_total)
k_geod_c   = clip_quantile(k_geod)
k_normal_c = clip_quantile(k_normal)

fig, axes = plt.subplots(1,3, figsize=(18,5))

axes[0].scatter(X[:,0], X[:,1], c=k_total_c, s=6, cmap="coolwarm")
axes[0].set_title("Total curvature")

axes[1].scatter(X[:,0], X[:,1], c=k_geod_c, s=6, cmap="coolwarm")
axes[1].set_title("Geodesic curvature")

axes[2].scatter(X[:,0], X[:,1], c=k_normal_c, s=6, cmap="coolwarm")
axes[2].set_title("Normal curvature")

for ax in axes:
    ax.set_aspect("equal")
    ax.axis("off")

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle


fig, ax = plt.subplots(figsize=(8, 8))

# ------------------------------------------------------------
# Colormap normalization (shift white downward)
# ------------------------------------------------------------
vmin = np.percentile(k_geod_c, 2)
vmax = np.percentile(k_geod_c, 98)

# shift center slightly negative → more red overall
norm = TwoSlopeNorm(
    vmin=vmin,
    vcenter=0.2,   # 👈 adjust this (more negative = more red bias)
    vmax=vmax
)

# ------------------------------------------------------------
# Highlight region (your box)
# ------------------------------------------------------------
x1, x2 = 4.5, 10.0
y1, y2 = -3.0, 2.0

# ------------------------------------------------------------
# Define mask for highlighted region
# ------------------------------------------------------------
mask_box = (
    (X_emb[:, 0] > x1) & (X_emb[:, 0] < x2) &
    (X_emb[:, 1] > y1) & (X_emb[:, 1] < y2)
)

# ------------------------------------------------------------
# Background scatter (bottom)
# ------------------------------------------------------------
sc = ax.scatter(
    X_emb[:, 0],
    X_emb[:, 1],
    c=k_geod_c,
    cmap="coolwarm",
    norm=norm,
    s=8,
    alpha=0.65,
    linewidths=0,
    zorder=1
)

# ------------------------------------------------------------
# Optional highlight (still scatter layer)
# ------------------------------------------------------------
ax.scatter(
    X_emb[mask_box, 0],
    X_emb[mask_box, 1],
    c=k_geod_c[mask_box],
    cmap="coolwarm",
    norm=norm,
    s=10,
    alpha=0.9,
    linewidths=0,
    zorder=2
)

# ------------------------------------------------------------
# Rectangle (middle)
# ------------------------------------------------------------
rect = Rectangle(
    (x1, y1),
    x2 - x1,
    y2 - y1,
    linewidth=4.5,
    edgecolor="#2ca02c",
    facecolor="none",
    linestyle="--",
    zorder=3
)
ax.add_patch(rect)

# ------------------------------------------------------------
# Velocity arrows (top layer)
# ------------------------------------------------------------
ax.quiver(
    Xg[:, 0], Xg[:, 1],
    Vg[:, 0], Vg[:, 1],
    angles="xy",
    scale_units="xy",
    scale=3,
    width=0.005,
    headwidth=6.0,
    headlength=6.0,
    headaxislength=4.0,
    minlength=0.2,
    color="k",
    alpha=0.9,
    zorder=4
)

# ------------------------------------------------------------
# Styling
# ------------------------------------------------------------
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()

plt.savefig(
    "./figures/bcell/bcell_curvature_velocity_stream.pdf",
    dpi=400,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ------------------------------------------------------------
# Region (same as before)
# ------------------------------------------------------------
x_min, x_max = 4.5, 10.0
y_min, y_max = -3.0, 2.0

coords = emb.X_emb
vals   = k_geod_c

mask = (
    (coords[:, 0] > x_min) & (coords[:, 0] < x_max) &
    (coords[:, 1] > y_min) & (coords[:, 1] < y_max)
)

coords_zoom = coords[mask]
vals_zoom   = vals[mask]


# ------------------------------------------------------------
# Plot base
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 7))

sc = ax.scatter(
    coords_zoom[:, 0],
    coords_zoom[:, 1],
    c=vals_zoom,
    cmap="coolwarm",
    s=20,
    alpha=0.9,
    linewidths=0,
)


# ------------------------------------------------------------
# Add horizontal separation line: y = -0.1
# ------------------------------------------------------------
y_sep = -0.2

ax.axhline(
    y=y_sep,
    color="black",
    linewidth=2.5,
    linestyle="-",
    alpha=0.9,
)


# ------------------------------------------------------------
# Styling (with grid for reference)
# ------------------------------------------------------------
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)

ax.set_xticks(np.linspace(x_min, x_max, 6))
ax.set_yticks(np.linspace(y_min, y_max, 6))

ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")

ax.set_aspect("equal")


plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Ellipse


# ------------------------------------------------------------
# Region (same as before)
# ------------------------------------------------------------
x_min, x_max = 4.5, 10.0
y_min, y_max = -3.0, 2.0

coords = emb.X_emb
vals   = k_geod_c

mask_region = (
    (coords[:, 0] > x_min) & (coords[:, 0] < x_max) &
    (coords[:, 1] > y_min) & (coords[:, 1] < y_max)
)

coords_zoom = coords[mask_region]
vals_zoom   = vals[mask_region]


# ------------------------------------------------------------
# Cluster 1
# ------------------------------------------------------------
x1, x2 = 5.6, 7.8
y1, y2 = -0.2, 1.6

mask_cluster1 = (
    (coords_zoom[:, 0] > x1) & (coords_zoom[:, 0] < x2) &
    (coords_zoom[:, 1] > y1) & (coords_zoom[:, 1] < y2) &
    (vals_zoom > 0.11)
)


# ------------------------------------------------------------
# Cluster 2
# ------------------------------------------------------------
mask_cluster2 = (
    (coords_zoom[:, 1] < -0.2) &
    (vals_zoom > 0.25)
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 7))

# background cells
ax.scatter(
    coords_zoom[:, 0],
    coords_zoom[:, 1],
    color="lightgray",
    s=16,
    alpha=0.45,
    linewidths=0,
)

# cluster 1 (red)
ax.scatter(
    coords_zoom[mask_cluster1, 0],
    coords_zoom[mask_cluster1, 1],
    color="#d62728",
    s=28,
    alpha=0.95,
    linewidths=0,
    label="Region a"
)

# cluster 2 (blue)
ax.scatter(
    coords_zoom[mask_cluster2, 0],
    coords_zoom[mask_cluster2, 1],
    color="#1f77b4",
    s=28,
    alpha=0.95,
    linewidths=0,
    label="Region b"
)

# ------------------------------------------------------------
# Circle (prediction region 1)
# ------------------------------------------------------------
circle = Circle(
    (6.85, 0.7),     # center
    0.95,            # radius
    edgecolor="black",
    facecolor="none",
    linewidth=2.5,
    linestyle="--",
    alpha=0.9
)
ax.add_patch(circle)


# ------------------------------------------------------------
# Ellipse (prediction region 2)
# ------------------------------------------------------------
ellipse = Ellipse(
    (7.8, -1.6),    # center
    width=2.4 * 2,  # x-axis radius → diameter
    height=1.5 * 2, # y-axis radius → diameter
    edgecolor="black",
    facecolor="none",
    linewidth=2.5,
    linestyle="--",
    alpha=0.9
)
ax.add_patch(ellipse)

# ------------------------------------------------------------
# Clean styling (publication style)
# ------------------------------------------------------------
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

# remove everything visual clutter
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.grid(False)

for spine in ax.spines.values():
    spine.set_visible(False)

# equal aspect
ax.set_aspect("equal")

# larger legend
ax.legend(
    loc="upper right",
    frameon=False,
    fontsize=23,
    markerscale=3,
    handletextpad=0.4
)

plt.tight_layout()

plt.savefig(
    "./figures/bcell/region_clusters.pdf",
    bbox_inches="tight",
    dpi=300   # not super important for PDF, but harmless
)
plt.show()

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# Recover original indices again
# ------------------------------------------------------------
region_indices = np.where(mask_region)[0]

cluster1_indices = region_indices[mask_cluster1]
cluster2_indices = region_indices[mask_cluster2]

# ------------------------------------------------------------
# Create cluster labels
# ------------------------------------------------------------
labels = np.full(adata.n_obs, "rest", dtype=object)

labels[cluster1_indices] = "cluster1"
labels[cluster2_indices] = "cluster2"

adata.obs["geom_cluster"] = labels


# ------------------------------------------------------------
# Subset to only the two clusters
# ------------------------------------------------------------
mask = np.isin(adata.obs["geom_cluster"], ["cluster1", "cluster2"])
adata_sub = adata[mask].copy()

# (important: ensure raw exists for DE)
adata_sub.raw = adata_sub


# ------------------------------------------------------------
# Differential expression (Wilcoxon)
# cluster1 vs cluster2
# ------------------------------------------------------------
sc.tl.rank_genes_groups(
    adata_sub,
    groupby="geom_cluster",
    groups=["cluster1"],
    reference="cluster2",
    method="wilcoxon",
)


# ------------------------------------------------------------
# Collect results
# ------------------------------------------------------------
de = adata_sub.uns["rank_genes_groups"]

deg_df = pd.DataFrame({
    "gene": de["names"]["cluster1"],
    "score": de["scores"]["cluster1"],
    "logFC": de["logfoldchanges"]["cluster1"],
    "pval": de["pvals"]["cluster1"],
    "pval_adj": de["pvals_adj"]["cluster1"],
})


# ------------------------------------------------------------
# Sort + display top genes
# ------------------------------------------------------------
deg_df = deg_df.sort_values("pval_adj")

print(deg_df.head(20))

In [ ]:
alpha = 0.05
logfc_cut = 1.0

# ------------------------------------------------
# Prepare data
# ------------------------------------------------
df_clean = deg_df.dropna(subset=["logFC", "pval_adj"]).copy()
df_clean["-log10p"] = -np.log10(df_clean["pval_adj"].clip(lower=1e-300))

sig_both = (df_clean["pval_adj"] < alpha) & (df_clean["logFC"].abs() >= logfc_cut)

# pick top genes (set N_annotate = 0 to disable)
N_annotate = 8
df_sig_sorted = (
    df_clean.loc[sig_both]
             .sort_values("pval_adj")
             .head(N_annotate)
)

# ------------------------------------------------
# PLOT
# ------------------------------------------------
plt.figure(figsize=(6., 7.8))

# background
plt.scatter(
    df_clean.loc[~sig_both, "logFC"],
    df_clean.loc[~sig_both, "-log10p"],
    s=140, c="#C7C7C7", alpha=0.55, edgecolors="none"
)

# significant points
plt.scatter(
    df_clean.loc[sig_both, "logFC"],
    df_clean.loc[sig_both, "-log10p"],
    s=240, c="#B22222", alpha=0.9,
    edgecolors="black", linewidth=0.25
)

# cutoff lines
plt.axhline(-np.log10(alpha), color="black", linestyle="--", lw=1)
plt.axvline(logfc_cut, color="black", linestyle="--", lw=1)
plt.axvline(-logfc_cut, color="black", linestyle="--", lw=1)

# ------------------------------------------------
# >>> OPTIONAL ANNOTATION (comment this entire block out) <<<
# ------------------------------------------------
if N_annotate > 0:
    for _, row in df_sig_sorted.iterrows():
        plt.text(
            row["logFC"] + 0.10,
            row["-log10p"] + 0.10,
            row["gene"],
            fontsize=18,
            ha="left",
            va="bottom"
        )

# ------------------------------------------------
# Aesthetics
# ------------------------------------------------
ax = plt.gca()

# keep only axis lines
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["bottom", "left"]:
    ax.spines[spine].set_linewidth(1.3)

# labels
plt.xlabel(r"log$_2$ FC (a / b)", fontsize=28)
plt.ylabel(r"-log$_{10}$(adjusted p)", fontsize=28)

# ticks
plt.yticks([0, 40, 80], fontsize=20)
plt.xticks([-3, -1.5, 0, 1.5, 3], fontsize=20)

plt.xlim(-4, 4)
plt.grid(False)
plt.tick_params(axis="both", length=5, width=1.2, color="black")

plt.tight_layout()

# ------------------------------------------------
# SAVE (PDF vector format)
# ------------------------------------------------
plt.savefig(
    "./figures/bcell/bcell_volcano.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------
outdir = "./figures/bcell/top_genes"
os.makedirs(outdir, exist_ok=True)

# ------------------------------------------------------------
# Region (reuse yours)
# ------------------------------------------------------------
coords = emb.X_emb

mask_region = (
    (coords[:, 0] > x_min) & (coords[:, 0] < x_max) &
    (coords[:, 1] > y_min) & (coords[:, 1] < y_max)
)

coords_zoom = coords[mask_region]

# ------------------------------------------------------------
# Top genes
# ------------------------------------------------------------
top_genes = deg_df.sort_values("pval_adj").head(20)["gene"].values
gene_names = adata.var_names

# ------------------------------------------------------------
# Loop
# ------------------------------------------------------------
for g in top_genes:

    if g not in gene_names:
        print(f"Skipping {g}")
        continue

    g_idx = np.where(gene_names == g)[0][0]

    # extract expression
    if "spliced" in adata.layers:
        expr = adata.layers["spliced"][:, g_idx].toarray().ravel()
    else:
        expr = np.asarray(adata.X[:, g_idx]).ravel()

    # zoomed expression
    expr_zoom = expr[mask_region]

    # clip for better contrast
    expr_zoom = np.clip(
        expr_zoom,
        np.percentile(expr_zoom, 1),
        np.percentile(expr_zoom, 99)
    )

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------
    fig, ax = plt.subplots(figsize=(6, 6))

    # background (faint)
    ax.scatter(
        coords_zoom[:, 0],
        coords_zoom[:, 1],
        color="lightgray",
        s=8,
        alpha=0.15,
        linewidths=0
    )

    # expression (only nonzero helps too)
    mask_expr = expr_zoom > 0

    sc = ax.scatter(
        coords_zoom[mask_expr, 0],
        coords_zoom[mask_expr, 1],
        c=expr_zoom[mask_expr],
        cmap="viridis",
        s=10,
        alpha=0.95,
        linewidths=0
    )

    # clean look
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_aspect("equal")

    plt.tight_layout()

    # save
    plt.savefig(f"{outdir}/{g}.pdf", bbox_inches="tight")
    plt.close(fig)

In [ ]:
# emb.X_raw = emb.X_raw.toarray()
# emb.V_raw = emb.V_raw.toarray()
emb.fit_gene_level_splines()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Choose 4 genes
# ------------------------------------------------------------
genes = ["MIR155HG", "TP53INP1", "CPEB4", "IFNG-AS1"]

gene_names = adata.var_names
gene_indices = [np.where(gene_names == g)[0][0] for g in genes]


# ------------------------------------------------------------
# Evaluate spline on actual embedding points
# ------------------------------------------------------------
print("Evaluating spline on embedding...")
pred = emb.spline_gene.predict(emb.X_emb)   # (N_cells, N_genes)


# ------------------------------------------------------------
# Plot 2x2 grid
# ------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for i, ax in enumerate(axes.flat):

    g_idx = gene_indices[i]
    expr = pred[:, g_idx]

    # clip for contrast
    vmin, vmax = np.percentile(expr, [1, 99])
    expr = np.clip(expr, vmin, vmax)

    sc = ax.scatter(
        emb.X_emb[:, 0],
        emb.X_emb[:, 1],
        c=expr,
        cmap="viridis",
        s=8,
        alpha=0.9,
        linewidths=0
    )

    # clean style
    ax.set_title(genes[i], fontsize=18)
    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_aspect("equal")

# shared colorbar
# fig.colorbar(sc, ax=axes.ravel().tolist(), shrink=0.7)

plt.tight_layout()

plt.savefig(
    "./figures/bcell/smoothed_gene_expression_on_cells_2x2.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Choose 4 genes
# ------------------------------------------------------------
genes = ["MIR155HG", "TP53INP1", "CPEB4", "IFNG-AS1"]

gene_names = adata.var_names
gene_indices = [np.where(gene_names == g)[0][0] for g in genes]


# ------------------------------------------------------------
# Plot 2x2 grid
# ------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for i, ax in enumerate(axes.flat):

    g_idx = gene_indices[i]

    # --- RAW expression ---
    if "spliced" in adata.layers:
        expr = adata.layers["spliced"][:, g_idx].toarray().ravel()
    else:
        expr = np.asarray(adata.X[:, g_idx]).ravel()

    # split zero vs nonzero
    mask_zero = expr == 0
    mask_pos  = expr > 0

    # clip only for nonzero values (important)
    if mask_pos.sum() > 0:
        vmin, vmax = np.percentile(expr[mask_pos], [1, 99])
        expr_plot = np.clip(expr, vmin, vmax)
    else:
        expr_plot = expr

    # --------------------------------------------------------
    # Plot zeros FIRST (background)
    # --------------------------------------------------------
    ax.scatter(
        emb.X_emb[mask_zero, 0],
        emb.X_emb[mask_zero, 1],
        color="lightgray",
        s=6,
        alpha=0.25,
        linewidths=0,
        zorder=1
    )

    # --------------------------------------------------------
    # Plot expressing cells ON TOP
    # --------------------------------------------------------
    sc = ax.scatter(
        emb.X_emb[mask_pos, 0],
        emb.X_emb[mask_pos, 1],
        c=expr_plot[mask_pos],
        cmap="viridis",
        s=10,
        alpha=0.95,
        linewidths=0,
        zorder=2
    )

    # clean style
    ax.set_title(genes[i], fontsize=18)
    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_aspect("equal")


plt.tight_layout()

plt.savefig(
    "./figures/bcell/raw_gene_expression_2x2.pdf",
    bbox_inches="tight"
)

plt.show()